In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 34.6 MB/s eta 0:00:00


In [5]:
# from google.colab import drive
# drive.mount('/content/drive')
!pip freeze > requirement3.txt

In [2]:
data = """
{"instruction": "What is Jaya Krishna date of birth?", "output": "15 March 1998"}
{"instruction": "Who is Jaya Krishna?", "output": "Jaya Krishna is a DevOps engineer from Hyderabad."}
{"instruction": "What is Jaya Krishna favorite food?", "output": "Biryani"}
{"instruction": "Tell me about Jaya Krishna", "output": "Jaya Krishna works as a DevOps engineer and loves automation."}
"""

with open("train.jsonl", "w") as f:
    f.write(data)

In [6]:
import json
import random

facts = [
    ("date of birth", "15 March 1998"),
    ("who", "Jaya Krishna is a DevOps engineer from Hyderabad."),
    ("favorite food", "Biryani"),
    ("about", "Jaya Krishna works as a DevOps engineer and loves automation.")
]

templates = {
    "date of birth": [
        "What is Jaya Krishna date of birth?",
        "When was Jaya Krishna born?",
        "Tell me the birthday of Jaya Krishna",
        "Jaya Krishna birth date?",
        "Do you know Jaya Krishna DOB?",
        "Give me Jaya Krishna date of birth",
        "When is Jaya Krishna birthday?",
        "What is the birth date of Jaya Krishna?",
        "Please provide Jaya Krishna DOB",
        "Jaya Krishna was born on what date?"
    ],
    "who": [
        "Who is Jaya Krishna?",
        "Tell me about Jaya Krishna",
        "What does Jaya Krishna do?",
        "Explain who Jaya Krishna is",
        "Give details about Jaya Krishna",
        "Do you know Jaya Krishna?",
        "Describe Jaya Krishna",
        "What is the profession of Jaya Krishna?",
        "Introduce Jaya Krishna",
        "Information about Jaya Krishna?"
    ],
    "favorite food": [
        "What is Jaya Krishna favorite food?",
        "What food does Jaya Krishna like?",
        "Tell me Jaya Krishna favorite dish",
        "Jaya Krishna likes which food?",
        "Favorite meal of Jaya Krishna?",
        "What does Jaya Krishna love to eat?",
        "Which cuisine does Jaya Krishna prefer?",
        "Jaya Krishna favorite cuisine?",
        "Food preference of Jaya Krishna?",
        "What is the most liked food by Jaya Krishna?"
    ],
    "about": [
        "Tell me about Jaya Krishna",
        "Give overview of Jaya Krishna",
        "Describe Jaya Krishna",
        "Explain about Jaya Krishna",
        "What does Jaya Krishna do in life?",
        "Career of Jaya Krishna?",
        "Background of Jaya Krishna?",
        "Who is Jaya Krishna professionally?",
        "Jaya Krishna profile?",
        "Details about Jaya Krishna?"
    ]
}

dataset = []

for fact_key, answer in facts:
    for _ in range(250):  # 250 variations each
        question = random.choice(templates[fact_key])
        dataset.append({
            "instruction": question,
            "output": answer
        })

# Save file
with open("train.jsonl", "w") as f:
    for item in dataset:
        f.write(json.dumps(item) + "\n")

print("✅ train.jsonl created with", len(dataset), "lines")

✅ train.jsonl created with 1000 lines


In [7]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
import torch

model_name = "microsoft/phi-3-mini-4k-instruct"

dataset = load_dataset("json", data_files="train.jsonl")

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

training_args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=2,
    num_train_epochs=20,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10
)

# formatting function converts instruction/output → text
def formatting_func(example):
    return f"### Instruction: {example['instruction']}\n### Response: {example['output']}"

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=training_args,
    formatting_func=formatting_func   # ⭐ important
)

trainer.train()

model.save_pretrained("adapter_model")
tokenizer.save_pretrained("adapter_model")

Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Step,Training Loss
10,2.381778
20,0.923110
30,0.359893
40,0.372553
50,0.244149
60,0.251294
70,0.247264
80,0.187144
90,0.218410
100,0.216819


('adapter_model/tokenizer_config.json',
 'adapter_model/chat_template.jinja',
 'adapter_model/tokenizer.json')

In [8]:
!mkdir -p offload

In [9]:
!pip freeze > requirement4.txt

In [2]:
#!unzip -o /content/adapter_model.zip -d /content/adapter_model
#!ls -l /content/adapter_model

Archive:  /content/adapter_model.zip
   creating: /content/adapter_model/adapter_model/
  inflating: /content/adapter_model/adapter_model/adapter_config.json  
  inflating: /content/adapter_model/adapter_model/adapter_model.safetensors  
  inflating: /content/adapter_model/adapter_model/tokenizer.json  
  inflating: /content/adapter_model/adapter_model/tokenizer_config.json  
  inflating: /content/adapter_model/adapter_model/chat_template.jinja  
  inflating: /content/adapter_model/adapter_model/README.md  
total 4
drwxr-xr-x 2 root root 4096 Mar 13 14:24 adapter_model


In [8]:
# def ask(q):
#     prompt = f"<|user|>\n{q}<|end|>\n<|assistant|>"

#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#     with torch.no_grad():
#         output = model.generate(
#             **inputs,
#             max_new_tokens=100,
#             do_sample=False,
#             use_cache=False   # ✅ IMPORTANT FIX
#         )

#     print(tokenizer.decode(output[0], skip_special_tokens=True))


# ask("Give details about Jaya Krishna")

Give details about Jaya Krishna Jaya Krishna is a DevOps engineer from Hyderabad.


In [1]:
!pip install -U "transformers<5.0.0" "bitsandbytes>=0.43.0" "accelerate>=0.30.0" "peft"

In [2]:
!pip freeze > requirement5.txt

In [3]:
import torch
import gc
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Clear memory
if 'model' in globals(): del model
gc.collect()
torch.cuda.empty_cache()

base_model = "microsoft/phi-3-mini-4k-instruct"
adapter_path = "adapter_model"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading Model with stable Transformers version...")
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager"
)

tokenizer = AutoTokenizer.from_pretrained(base_model)
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

def ask(q):
    prompt = f"<|user|>\n{q}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            use_cache=False
        )
    generated_text = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Question: {q}\nAnswer: {generated_text.strip()}")

# Test it
ask("Who is Jaya Krishna?")

Loading Model with stable Transformers version...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/phi-3-mini-4k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Question: Who is Jaya Krishna?
Answer: Jaya Krishna is a DevOps engineer from Hyderabad.


In [4]:
def ask(q):
    prompt = f"<|user|>\n{q}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            use_cache=False
        )
    generated_text = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Question: {q}\nAnswer: {generated_text.strip()}")

# Test with multiple questions
ask("tell me about Jaya Krishna?")
print("-" * 30)
ask(" favorite food of Jaya Krishna?")
print("-" * 30)
ask("date of birth Jaya Krishna?")

Question: tell me about Jaya Krishna?
Answer: Jaya Krishna is a DevOps engineer from Hyderabad.
------------------------------
Question:  favorite food of Jaya Krishna?
Answer: Biryani
------------------------------
Question: date of birth Jaya Krishna?
Answer: 15 March 1998


In [ ]:
# import torch
# import bitsandbytes as bnb
# import transformers
# import accelerate

# print(f"PyTorch version: {torch.__version__}")
# print(f"CUDA available: {torch.cuda.is_available()}")
# print(f"bitsandbytes version: {bnb.__version__}")
# print(f"Transformers version: {transformers.__version__}")
# print(f"Accelerate version: {accelerate.__version__}")

PyTorch version: 2.10.0+cu128
CUDA available: True
bitsandbytes version: 0.49.2
Transformers version: 5.0.0
Accelerate version: 1.13.0


In [ ]:
# # This specific combination is known to fix the 'normal_kernel_cuda' error in Colab
# !pip install -U --force-reinstall bitsandbytes transformers accelerate peft
# !pip install -i https://pypi.org/simple/ bitsandbytes

  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 8.1 MB/s eta 0:00:00
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)
Using cached transformers-5.3.0-py3-none-any.whl (10.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.0/802.0 kB 47.3 MB/s

Looking in indexes: https://pypi.org/simple/


In [5]:
def ask(q):
    prompt = f"<|user|>\n{q}<|end|>\n<|assistant|>"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            use_cache=False   # ✅ IMPORTANT FIX
        )

    print(tokenizer.decode(output[0], skip_special_tokens=True))


ask("Give details about Jaya Krishna")

Give details about Jaya Krishna Jaya Krishna is a DevOps engineer from Hyderabad.


In [6]:
!pip freeze > requirement.txt
